In [0]:
from pyspark.sql.functions import col, expr, lit, when

silver_municipality_df = spark.table("silver.resultado_municipio")
silver_state_df         = spark.table("silver.resultado_uf")
silver_brazil_df        = spark.table("silver.resultado_brasil")

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")


#Literacy indicator by municipality
gold_indicator_municipality_df = silver_municipality_df.select(
    "ano", "id_municipio", "municipio_nome", "nome_uf", "nome_regiao",
    "amazonia_legal", "rede", "taxa_alfabetizacao", "taxa_alfabetizacao_recalculada",
    "possui_meta_municipal", "meta_alfabetizacao_2030", "centroide"
).withColumn(
    "gap_to_2030_goal",
    col("meta_alfabetizacao_2030") - col("taxa_alfabetizacao")
)

gold_indicator_municipality_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.indicador_alfabetizacao_municipio")

#Goal vs. result comparison
comparison_municipality_df = silver_municipality_df.select(
    lit("Município").alias("nivel_geografico"),
    col("id_municipio").alias("identificador"),
    col("municipio_nome").alias("nome"),
    "ano", "rede", "taxa_alfabetizacao",
    col("taxa_alfabetizacao_meta_base").alias("meta_base"),
    "meta_alfabetizacao_2030"
)

comparison_state_df = silver_state_df.select(
    lit("UF").alias("nivel_geografico"),
    col("sigla_uf").alias("identificador"),
    col("sigla_uf").alias("nome"),
    "ano", "rede", "taxa_alfabetizacao",
    col("taxa_alfabetizacao_meta_base").alias("meta_base"),
    "meta_alfabetizacao_2030"
)

comparison_brazil_df = silver_brazil_df.select(
    lit("Brasil").alias("nivel_geografico"),
    lit("BR").alias("identificador"),
    lit("Brasil").alias("nome"),
    "ano", "rede", "taxa_alfabetizacao",
    lit(None).cast("double").alias("meta_base"),
    "meta_alfabetizacao_2030"
)

gold_comparison_df = (comparison_municipality_df
    .unionByName(comparison_state_df)
    .unionByName(comparison_brazil_df)
    .withColumn("gap_to_2030_goal", col("meta_alfabetizacao_2030") - col("taxa_alfabetizacao")))

gold_comparison_df.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.comparativo_meta_resultado")

#Indicator evolution over time
actual_df = (silver_municipality_df
    .groupBy("ano", "nome_uf", "nome_regiao")
    .agg(expr("avg(taxa_alfabetizacao) as valor"))
    .withColumn("tipo", lit("realizado")))

# unpivot the goal columns (2024..2030) into ano/valor rows
goal_unpivot_df = silver_municipality_df.select(
    "nome_uf", "nome_regiao",
    expr("""stack(7,
        2024, meta_alfabetizacao_2024,
        2025, meta_alfabetizacao_2025,
        2026, meta_alfabetizacao_2026,
        2027, meta_alfabetizacao_2027,
        2028, meta_alfabetizacao_2028,
        2029, meta_alfabetizacao_2029,
        2030, meta_alfabetizacao_2030
    ) as (ano, valor)""")
).groupBy("ano", "nome_uf", "nome_regiao").agg(expr("avg(valor) as valor")) \
 .withColumn("tipo", lit("meta"))

gold_time_evolution_df = actual_df.unionByName(goal_unpivot_df)

gold_time_evolution_df.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.evolucao_temporal_indicador")

print("Gold layer written: indicador_alfabetizacao_municipio, comparativo_meta_resultado, evolucao_temporal_indicador")